# Gemini Setup

In [1]:
from google import genai

client = genai.Client()

MODEL_NAME = "gemini-3.1-flash-lite-preview"

## Simple first request

In [9]:
text = "Hey, are you down to grab some pizza later? I'm starving!"

response = client.models.generate_content(
    model=MODEL_NAME,
    config={
        "system_instruction": "Only output the translated text"
    },
    contents=f"Translate the following text to Dutch: {text}"
)

print(response.text)

Hey, heb je zin om straks pizza te gaan eten? Ik verga van de honger!


In [14]:
print(response.model_dump_json(indent=2))

{
  "sdk_http_response": {
    "headers": {
      "x-gemini-service-tier": "standard",
      "content-type": "application/json; charset=UTF-8",
      "vary": "Origin, X-Origin, Referer",
      "content-encoding": "gzip",
      "date": "Fri, 17 Apr 2026 13:19:38 GMT",
      "server": "scaffolding on HTTPServer2",
      "x-xss-protection": "0",
      "x-frame-options": "SAMEORIGIN",
      "x-content-type-options": "nosniff",
      "server-timing": "gfet4t7; dur=6852",
      "alt-svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000",
      "transfer-encoding": "chunked"
    },
    "body": null
  },
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "media_resolution": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": null,
            "function_response": null,
            "inline_data": null,
            "text": "Hey, heb je zin om straks pizza te gaan e

In [13]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=30,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=30
    ),
  ],
  total_token_count=50
)

In [18]:
message = "tell me a bad time story about a unicorn"

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=message,
)

print(response.text)

"Once upon a time, in a forest where the trees whispered secrets you were never meant to hear, lived a unicorn named Barnaby. \n\nBarnaby was not the kind of unicorn you see on sparkly lunchboxes. He didn’t have a coat like spun silk; his fur was matted, smelling faintly of swamp water and old, wet pennies. His horn wasn’t a shimmering spire of magic, but a jagged, splintered piece of bone that looked like it had been sharpened against a tombstone.\n\nWhile other unicorns spent their days chasing rainbows, Barnaby spent his nights watching you. \n\nHe didn't move like a normal horse. He didn't trot or gallop. He moved with a jerky, unnatural rhythm, like a marionette being pulled by strings tied to your own pulse. *Click-clack, click-clack* went his hooves against the hardwood floor outside your bedroom door. It was a rhythmic, hollow sound, the sound of something that didn't quite belong in this world.\n\nBarnaby didn’t eat clover. He ate dreams. Not the sweet, fluffy ones about flyin

In [19]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=519,
  prompt_token_count=10,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
    ),
  ],
  total_token_count=529
)

## Understanding costs

In [20]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5-nano": {"input": 0.075, "output": 0.30},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5.2": {"input": 1.75, "output": 14.00},
    "gpt-5.2-pro": {"input": 21.00, "output": 168.00},
    "gemini-3.1-flash-lite-preview": {"input": 0.25 , "output": 1.50},
}

In [21]:
def calculate_cost(model_name: str, input_tokens: int, output_tokens: int) -> float:
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost

In [23]:
usage = response.usage_metadata

cost = calculate_cost(
    model_name=MODEL_NAME,
    input_tokens=usage.prompt_token_count,
    output_tokens=usage.candidates_token_count
)

print(f"Cost: ${cost:.6f}")

Cost: $0.000781


## Streaming requests

In [31]:
message = 'Tell me a funny joke about trains.'

stream = client.models.generate_content_stream(
    model=MODEL_NAME,
    contents=message,
)

for chunk in stream:
    print(chunk.text)

Why do train conductors make terrible comedians?

Because they have no
 sense of **loco-motive**, and they always lose their **train of thought!**



## LLMs are stateless

In [ ]:
instructions = "make sure to add emojis"
chat = client.chats.create(model=MODEL_NAME, config={"system_instruction": instructions})

response = chat.send_message("My name is Lars and I like to eat pizza.")
print(response.text)

response = chat.send_message("What is my name?")
print(response.text)

for message in chat.get_history():
    print(f'role - {message.role}',end=": ")
    print(message.parts[0].text)


Nice to meet you, Lars! 🍕 It’s great to meet a fellow pizza lover. There’s really nothing quite like a delicious slice, is there? 😋

What’s your favorite kind of pizza? Are you a classic pepperoni fan, or do you like to get a little adventurous with your toppings? 🍕✨
Your name is Lars! 😊🍕
role - user: My name is Lars and I like to eat pizza.
role - model: Nice to meet you, Lars! 🍕 It’s great to meet a fellow pizza lover. There’s really nothing quite like a delicious slice, is there? 😋

What’s your favorite kind of pizza? Are you a classic pepperoni fan, or do you like to get a little adventurous with your toppings? 🍕✨
role - user: What is my name?
role - model: Your name is Lars! 😊🍕
